<a href="https://colab.research.google.com/github/awinarko-hue/Project_hue/blob/main/Legal/GraphRAG_Komdigi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Graph-RAG untuk Sistem Penelusuran & Penalaran Regulasi Komdigi
**Knowledge Graph + Large Language Model — Penelitian S2 Magister Informatika**

Notebook ini mengimplementasikan roadmap 6 tahap:

| Tahap | Isi | Bagian Notebook |
|---|---|---|
| 1 | Persiapan data: pembersihan, standarisasi metadata, validasi referensi | §1 |
| 2 | Baseline RAG: chunking → embedding → FAISS → LLM | §2 |
| 3 | Hybrid Retrieval: dense + BM25 + filter metadata (RRF) | §3 |
| 4 | Knowledge Graph: dokumen→bab→pasal→ayat, rujukan silang, entitas | §4 |
| 5 | Graph RAG: semantic retrieval + traversal graf | §5 |
| 6 | Evaluasi: Recall@K, Precision@K, MRR, Hit Rate (+ opsi RAGAS) | §6 |

**Desain untuk korpus yang terus bertambah:** semua file `korpus*.csv` dalam folder korpus
akan dimuat otomatis. Cukup tambahkan file CSV baru dengan skema kolom yang sama
(`doc_id, nama_dokumen, jenis, tahun, nomor, bab, nama_bab, bagian, paragraf, pasal, ayat, huruf, angka, tipe, isi, referensi, kata_kunci, entitas`)
lalu jalankan ulang §1, §2 (indexing), dan §4 (graf).

## §0 — Setup Lingkungan

In [1]:
# Instalasi dependensi (sekali per sesi Colab)
!pip install -q sentence-transformers faiss-cpu rank_bm25 networkx pyvis google-generativeai

import os, re, glob, json, time, pickle
import numpy as np
import pandas as pd
import networkx as nx
pd.set_option('display.max_colwidth', 120)
print("Setup selesai.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 26.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 756.0/756.0 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 21.3 MB/s eta 0:00:00
Setup selesai.


In [2]:
MOUNT_DRIVE = True

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR   = '/content/drive/MyDrive/Tesis_SDPPI/Project 7/GraphRAG_Komdigi'   # sesuaikan
else:
    BASE_DIR   = '/content/GraphRAG_Komdigi'

KORPUS_DIR   = os.path.join(BASE_DIR, '/content/drive/MyDrive/Tesis_SDPPI/Project 7/Aplikasi_Skema/Korpus Komdigi')
IFACT_DIR = os.path.join(BASE_DIR, 'artifacts')   # indeks FAISS, graf, embedding
os.makedirs(KORPUS_DIR, exist_ok=True)
os.makedirs(IFACT_DIR, exist_ok=True) # Changed ARTIFACT_DIR to IFACT_DIR

# Model embedding multilingual (konsisten dengan pipeline FAISS Anda sebelumnya).
# Alternatif lebih akurat namun lebih berat: 'intfloat/multilingual-e5-base'
EMBED_MODEL_NAME = 'paraphrase-multilingual-MiniLM-L12-v2'

print("Korpus dicari di:", KORPUS_DIR)
print("File terdeteksi :", sorted(os.path.basename(f) for f in glob.glob(os.path.join(KORPUS_DIR, 'korpus*.csv'))))

Mounted at /content/drive
Korpus dicari di: /content/drive/MyDrive/Tesis_SDPPI/Project 7/Aplikasi_Skema/Korpus Komdigi
File terdeteksi : ['korpusPM172018.csv', 'korpusPM42025.csv', 'korpusPM52021.csv', 'korpusPM82024.csv', 'korpusPM82026.csv', 'korpusPM92023.csv', 'korpusPM_2_2023.csv', 'korpusPM_2_2025.csv', 'korpusPP282025.csv', 'korpusPP462021.csv', 'korpusPP492005.csv', 'korpusPP512002.csv', 'korpusPP522000.csv', 'korpusPP532000.csv', 'korpusPerppu22022.csv', 'korpusPerpres_174_2024.csv', 'korpusUU122011.csv', 'korpusUU322002.csv', 'korpusUU361999.csv', 'korpusUU382009.csv', 'korpusUU62023.csv']


> **Catatan:** Jika baru pertama kali, unggah/salin isi `Korpus_Komdigi.zip` ke folder `korpus` di Drive.
> Sel di bawah ini opsional — hanya jika Anda ingin unggah ZIP langsung ke sesi Colab.

In [ ]:
# (Opsional) Unggah ZIP korpus langsung ke sesi ini
# from google.colab import files
# up = files.upload()                      # pilih Korpus_Komdigi.zip
# !unzip -o -q Korpus_Komdigi.zip -d /content/_zip
# !cp /content/_zip/*/korpus*.csv "$KORPUS_DIR"/
# print(sorted(os.listdir(KORPUS_DIR)))

## §1 — Tahap 1: Persiapan Data
Memuat seluruh `korpus*.csv`, membersihkan teks, menstandarkan metadata, membentuk
`chunk_id` unik per satuan hukum, dan memvalidasi referensi antar-pasal.

In [ ]:
SCHEMA = ['doc_id','nama_dokumen','jenis','tahun','nomor','bab','nama_bab','bagian',
          'paragraf','pasal','ayat','huruf','angka','tipe','isi','referensi','kata_kunci','entitas']

def load_korpus(korpus_dir: str) -> pd.DataFrame:
    frames, laporan = [], []
    for path in sorted(glob.glob(os.path.join(korpus_dir, 'korpus*.csv'))):
        df = pd.read_csv(path)
        hilang = [c for c in SCHEMA if c not in df.columns]
        if hilang:
            print(f"[LEWATI] {os.path.basename(path)} — kolom hilang: {hilang}")
            continue
        df = df[SCHEMA].copy()
        df['sumber_file'] = os.path.basename(path)
        frames.append(df)
        laporan.append((os.path.basename(path), len(df), df.doc_id.iloc[0]))
    korpus = pd.concat(frames, ignore_index=True)
    print(pd.DataFrame(laporan, columns=['file','baris','doc_id']).to_string(index=False))
    return korpus

def bersihkan(korpus: pd.DataFrame) -> pd.DataFrame:
    df = korpus.copy()
    # 1. Normalisasi teks
    df['isi'] = (df['isi'].astype(str)
                   .str.replace(r'\s+', ' ', regex=True)
                   .str.strip())
    df = df[df['isi'].str.len() > 0].copy()
    # 2. Standarisasi metadata
    df['jenis'] = df['jenis'].astype(str).str.strip()
    df['tahun'] = pd.to_numeric(df['tahun'], errors='coerce').astype('Int64')
    for c in ['pasal','ayat','angka']:
        df[c] = (df[c].astype('string').str.strip()
                       .str.replace(r'\.0$', '', regex=True))   # '3.0' → '3'
    df['huruf'] = df['huruf'].astype('string').str.strip().str.lower()
    # 3. Buang duplikat persis
    n0 = len(df)
    df = df.drop_duplicates(subset=['doc_id','pasal','ayat','huruf','angka','isi'])
    if n0 - len(df): print(f"Duplikat dibuang: {n0-len(df)}")
    # 4. chunk_id unik & hirarkis, mis. UU_6_2023__P154A__a1__h-g__n4
    def buat_id(r):
        parts = [str(r['doc_id']), f"P{r['pasal']}" if pd.notna(r['pasal']) else "P-"]
        if pd.notna(r['ayat']):  parts.append(f"a{r['ayat']}")
        if pd.notna(r['huruf']): parts.append(f"h{r['huruf']}")
        if pd.notna(r['angka']): parts.append(f"n{r['angka']}")
        return "__".join(parts)
    df['chunk_id'] = df.apply(buat_id, axis=1)
    # jika masih ada bentrok id (unit sama terpecah beberapa baris), beri sufiks urut
    dup = df.duplicated('chunk_id', keep=False)
    df.loc[dup, 'chunk_id'] = (df.loc[dup, 'chunk_id'] + '__' +
                               df.loc[dup].groupby('chunk_id').cumcount().astype(str))
    df = df.reset_index(drop=True)
    return df

korpus = bersihkan(load_korpus(KORPUS_DIR))
print(f"\nTotal satuan hukum: {len(korpus)} | Dokumen: {korpus.doc_id.nunique()}")
korpus.head(3)

                      file  baris           doc_id
        korpusPM172018.csv    630   Permen_17_2018
         korpusPM42025.csv      5    Permen_4_2025
         korpusPM52021.csv   1463    Permen_5_2021
         korpusPM82024.csv    246    Permen_8_2024
         korpusPM82026.csv    418    Permen_8_2026
         korpusPM92023.csv    436    Permen_9_2023
       korpusPM_2_2023.csv    323    Permen_2_2023
       korpusPM_2_2025.csv    163    Permen_2_2025
        korpusPP282025.csv   2560       PP_28_2025
        korpusPP462021.csv    611       PP_46_2021
        korpusPP492005.csv     21       PP_49_2005
        korpusPP512002.csv    318       PP_51_2005
        korpusPP522000.csv    417       PP_52_2000
        korpusPP532000.csv    177       PP_53_2000
     korpusPerppu22022.csv   6607    Perppu_2_2022
korpusPerpres_174_2024.csv    123 Perpres_174_2024
        korpusUU122011.csv    981       UU_12_2011
        korpusUU322002.csv    399       UU_32_2002
        korpusUU361999.csv    2

,doc_id,nama_dokumen,jenis,tahun,nomor,bab,nama_bab,bagian,paragraf,pasal,ayat,huruf,angka,tipe,isi,referensi,kata_kunci,entitas,sumber_file,chunk_id
0,Permen_17_2018,Permen Nomor 17 Tahun 2018,Permen,2018,17,BAB I,Ketentuan Umum,NaN,NaN,1,<NA>,<NA>,<NA>,Definisi,Dalam Peraturan Menteri ini yang dimaksud dengan:,Pasal 1,menteri,Menteri,korpusPM172018.csv,Permen_17_2018__P1
1,Permen_17_2018,Permen Nomor 17 Tahun 2018,Permen,2018,17,BAB I,Ketentuan Umum,NaN,NaN,1,<NA>,<NA>,1,Ketentuan,Komunikasi Radio adalah telekomunikasi dengan mempergunakan gelombang radio.,Pasal 1 angka 1,"radio, komunikasi, telekomunikasi, mempergunakan, gelombang",NaN,korpusPM172018.csv,Permen_17_2018__P1__n1
2,Permen_17_2018,Permen Nomor 17 Tahun 2018,Permen,2018,17,BAB I,Ketentuan Umum,NaN,NaN,1,<NA>,<NA>,2,Ketentuan,"Kegiatan Amatir Radio adalah Komunikasi Radio mengenai ilmu pengetahuan, penyelidikan teknis dan informasi yang berk...",Pasal 1 angka 2,"radio, kegiatan, amatir, komunikasi, mengenai",NaN,korpusPM172018.csv,Permen_17_2018__P1__n2


In [ ]:
# ── Validasi referensi ─────────────────────────────────────────────
# a) Konsistensi lokator: kolom `referensi` vs metadata pasal/ayat baris tsb.
# b) Ekstraksi rujukan silang dari teks `isi`:
#    - internal : "Pasal 12 ayat (3)"  → pasal lain di dokumen yang sama
#    - eksternal: "Undang-Undang Nomor 36 Tahun 1999" → dokumen lain

POLA_INTERNAL  = re.compile(r'Pasal\s+(\d+[A-Z]*)(?:\s+ayat\s*\((\d+)\))?', re.I)
POLA_EKSTERNAL = re.compile(
    r'(Undang-Undang|Peraturan\s+Pemerintah(?:\s+Pengganti\s+Undang-Undang)?|'
    r'Peraturan\s+Presiden|Peraturan\s+Menteri)\s+Nomor\s+(\d+)\s+Tahun\s+(\d{4})', re.I)

JENIS_MAP = {'undang-undang':'UU', 'peraturan pemerintah':'PP',
             'peraturan pemerintah pengganti undang-undang':'Perppu',
             'peraturan presiden':'Perpres', 'peraturan menteri':'Permen'}

def ekstrak_rujukan(df: pd.DataFrame):
    internal, eksternal = [], []
    doc_lookup = {}   # (jenis, nomor, tahun) -> doc_id
    for _, r in df.drop_duplicates('doc_id').iterrows():
        doc_lookup[(str(r['jenis']).lower(), str(r['nomor']), str(r['tahun']))] = r['doc_id']
    pasal_ada = set(zip(df['doc_id'], df['pasal'].astype(str)))
    for _, r in df.iterrows():
        teks = r['isi']
        for m in POLA_INTERNAL.finditer(teks):
            p_tuju = m.group(1)
            if str(p_tuju) != str(r['pasal']):     # abaikan rujukan ke diri sendiri
                internal.append({'chunk_id': r['chunk_id'], 'doc_id': r['doc_id'],
                                 'pasal_asal': r['pasal'], 'pasal_tuju': p_tuju,
                                 'ayat_tuju': m.group(2),
                                 'valid': (r['doc_id'], p_tuju) in pasal_ada})
        for m in POLA_EKSTERNAL.finditer(teks):
            jenis_key = re.sub(r'\s+', ' ', m.group(1).lower())
            jenis = JENIS_MAP.get(jenis_key, m.group(1))
            key = (jenis.lower(), m.group(2), m.group(3))
            tuju = None
            for (j, n, t), did in doc_lookup.items():
                if n == m.group(2) and t == m.group(3):
                    tuju = did; break
            eksternal.append({'chunk_id': r['chunk_id'], 'doc_id': r['doc_id'],
                              'jenis_tuju': jenis, 'nomor': m.group(2), 'tahun': m.group(3),
                              'doc_tuju': tuju, 'dalam_korpus': tuju is not None})
    return pd.DataFrame(internal), pd.DataFrame(eksternal)

ref_internal, ref_eksternal = ekstrak_rujukan(korpus)

print("=== Rujukan internal antar-pasal ===")
print(f"Total: {len(ref_internal)} | valid (pasal tujuan ada): {ref_internal['valid'].sum()}"
      f" | tak ditemukan: {(~ref_internal['valid']).sum()}")
print("\n=== Rujukan antar-dokumen ===")
print(f"Total sebutan: {len(ref_eksternal)} | menuju dokumen dalam korpus: {ref_eksternal['dalam_korpus'].sum()}")
print("\nRegulasi luar korpus yang paling sering dirujuk (kandidat perluasan korpus):")
print(ref_eksternal[~ref_eksternal.dalam_korpus]
      .groupby(['jenis_tuju','nomor','tahun']).size().sort_values(ascending=False).head(10))

=== Rujukan internal antar-pasal ===
Total: 6744 | valid (pasal tujuan ada): 6426 | tak ditemukan: 318

=== Rujukan antar-dokumen ===
Total sebutan: 221 | menuju dokumen dalam korpus: 41

Regulasi luar korpus yang paling sering dirujuk (kandidat perluasan korpus):
jenis_tuju  nomor  tahun
UU          3      1989     7
            11     2020     7
            36     2000     6
            7      1992     6
            25     1992     4
            21     2008     4
Perppu      1      2000     4
UU          2      1981     4
            5      1997     4
            41     1999     4
dtype: int64


In [ ]:
korpus['nomor'] = pd.to_numeric(korpus['nomor'], errors='coerce').astype('Int64') # menunjukkan bahwa kolom nomor dalam DataFrame korpus Anda memiliki data yang tidak dapat dikonversi ke tipe integer saat menyimpan ke format Parquet.

korpus.to_parquet(os.path.join(IFACT_DIR, 'korpus_bersih.parquet'))
ref_internal.to_csv(os.path.join(IFACT_DIR, 'ref_internal.csv'), index=False)
ref_eksternal.to_csv(os.path.join(IFACT_DIR, 'ref_eksternal.csv'), index=False)
print("Artefak Tahap 1 tersimpan di", IFACT_DIR)

Artefak Tahap 1 tersimpan di /content/drive/MyDrive/Tesis_SDPPI/Project 7/GraphRAG_Komdigi/artifacts


## §2 — Tahap 2: Baseline RAG
Chunking sudah alami mengikuti struktur hukum (1 baris = 1 satuan pasal/ayat/huruf/angka).
Setiap chunk diberi *header* konteks (nama dokumen + posisi) agar embedding lebih diskriminatif.

In [ ]:
from sentence_transformers import SentenceTransformer
import faiss

def teks_chunk(r) -> str:
    lokasi = f"Pasal {r['pasal']}"
    if pd.notna(r['ayat']):  lokasi += f" ayat ({r['ayat']})"
    if pd.notna(r['huruf']): lokasi += f" huruf {r['huruf']}"
    if pd.notna(r['angka']): lokasi += f" angka {r['angka']}"
    return f"{r['nama_dokumen']} — {r['nama_bab']} — {lokasi} [{r['tipe']}]: {r['isi']}"

korpus['teks_embed'] = korpus.apply(teks_chunk, axis=1)

embedder = SentenceTransformer(EMBED_MODEL_NAME)

EMB_PATH = os.path.join(IFACT_DIR, 'embeddings.npy')
IDX_PATH = os.path.join(IFACT_DIR, 'faiss.index')

def bangun_indeks(force=False):
    if (not force) and os.path.exists(EMB_PATH) and os.path.exists(IDX_PATH):
        emb = np.load(EMB_PATH)
        if len(emb) == len(korpus):           # korpus belum berubah → pakai cache
            return emb, faiss.read_index(IDX_PATH)
        print("Ukuran korpus berubah — indeks dibangun ulang.")
    emb = embedder.encode(korpus['teks_embed'].tolist(), batch_size=64,
                          show_progress_bar=True, normalize_embeddings=True)
    emb = np.asarray(emb, dtype='float32')
    index = faiss.IndexFlatIP(emb.shape[1])   # inner product = cosine (sudah dinormalisasi)
    index.add(emb)
    np.save(EMB_PATH, emb); faiss.write_index(index, IDX_PATH)
    return emb, index

embeddings, faiss_index = bangun_indeks()
print("Indeks FAISS:", faiss_index.ntotal, "vektor, dim", embeddings.shape[1])

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Ukuran korpus berubah — indeks dibangun ulang.


Batches:   0%|          | 0/380 [00:00<?, ?it/s]

Indeks FAISS: 24294 vektor, dim 384


In [ ]:
def retrieve_dense(query: str, k: int = 10) -> pd.DataFrame:
    qv = embedder.encode([query], normalize_embeddings=True).astype('float32')
    skor, idx = faiss_index.search(qv, k)
    hasil = korpus.iloc[idx[0]].copy()
    hasil['skor'] = skor[0]
    return hasil[['chunk_id','nama_dokumen','pasal','ayat','tipe','isi','skor']]

retrieve_dense("sanksi bagi penyelenggara telekomunikasi tanpa izin", k=5)

,chunk_id,nama_dokumen,pasal,ayat,tipe,isi,skor
15878,UU_36_1999__P21__0,UU Nomor 36 Tahun 1999,21,<NA>,Larangan,Penyelenggara telekomunikasi dilarang melakukan kegiatan usaha penyelenggaraan telekomunikasi yang bertentangan deng...,0.823575
1023,Permen_5_2021__P48,Permen Nomor 5 Tahun 2021,48,<NA>,Larangan,"Penyelenggara Telekomunikasi dilarang melakukan penerapan tarif yang mengganggu perlindungan konsumen, persaingan us...",0.816534
7256,PP_52_2000__P33__0,PP Nomor 52 Tahun 2000,33,<NA>,Sanksi,Setiap penyelenggara jaringan atau penyelenggara jasa telekomunikasi yang tidak atau terlambat membayar Biaya Hak Pe...,0.780494
1739,Permen_5_2021__P175,Permen Nomor 5 Tahun 2021,175,<NA>,Sanksi,Penyelenggara Jasa Telekomunikasi wajib mensosialisasikan melalui berbagai media dan saluran Telekomunikasi peratura...,0.774846
16001,UU_36_1999__P29,UU Nomor 36 Tahun 1999,29,<NA>,Ketentuan,Ayat (1) Larangan bagi penyelenggara telekomunikasi khusus untuk disambungkan ke jaringan penyelenggara telekomunika...,0.769683


In [ ]:
# ── LLM: Gemini (gratis di Colab) — bisa diganti API lain ─────────
import google.generativeai as genai
from google.colab import userdata

USE_LLM = True
try:
    genai.configure(api_key=userdata.get('GEMINI_API_KEY'))   # simpan di Secrets Colab (ikon kunci)
    llm = genai.GenerativeModel('gemini-3.6-flash') # Diperbarui ke gemini-3.6-flash sesuai saran error
except Exception as e:
    USE_LLM = False
    print("LLM nonaktif (API key belum diset). Jawaban akan berupa konteks mentah.\n", e)

PROMPT_RAG = """Anda adalah asisten hukum untuk regulasi Kementerian Komunikasi dan Digital (Komdigi).
Jawab pertanyaan HANYA berdasarkan konteks regulasi berikut. Selalu sebutkan dasar hukumnya
(nama peraturan, pasal, ayat). Jika konteks tidak memadai, katakan tidak ditemukan.

KONTEKS:
{konteks}

PENNYATAAN: {pertanyaan}

JAWABAN (bahasa Indonesia formal, dengan sitasi pasal):"""

def rakit_konteks(df_hasil: pd.DataFrame) -> str:
    baris = []
    for _, r in df_hasil.iterrows():
        lok = f"Pasal {r['pasal']}" + (f" ayat ({r['ayat']})" if pd.notna(r.get('ayat')) else "")
        baris.append(f"[{r.get('nama_dokumen', r['chunk_id'])} — {lok}] {r['isi']}")
    return "\n\n".join(baris)

def jawab(pertanyaan: str, retriever, k: int = 8):
    t0 = time.time()
    hasil = retriever(pertanyaan, k)
    t_ret = time.time() - t0
    konteks = rakit_konteks(hasil)
    if USE_LLM:
        t0 = time.time()
        out = llm.generate_content(PROMPT_RAG.format(konteks=konteks, pertanyaan=pertanyaan)).text
        t_llm = time.time() - t0
    else:
        out, t_llm = "(LLM nonaktif — konteks teratas:)\n" + konteks[:1500], 0.0
    return {'jawaban': out, 'konteks': hasil, 'waktu_retrieval': t_ret, 'waktu_llm': t_llm}

res = jawab("Apa kewajiban penyelenggara amatir radio?", retrieve_dense)
print(res['jawaban'])
print(f"\n[retrieval {res['waktu_retrieval']:.2f}s | LLM {res['waktu_llm']:.2f}s]")

Berdasarkan **Permen Nomor 17 Tahun 2018 Pasal 40 ayat (1)**, kewajiban Amatir Radio adalah menjamin pancaran Komunikasi Radionya tidak mengganggu atau menimbulkan interferensi yang merugikan terhadap Kegiatan Amatir Radio lainnya dan/atau komunikasi radio dinas lain.

[retrieval 0.04s | LLM 26.87s]


LLM yang lain

In [ ]:
# — LLM: GPT-OSS 120B via Groq (Compatible OpenAI, GRATIS) —
# Catatan: llama-3.3-70b-versatile sudah di-deprecate Groq per 16 Agustus 2026.
# Model ini adalah pengganti resmi yang direkomendasikan Groq.
import os
import time
import pandas as pd
from openai import OpenAI
from google.colab import userdata

USE_LLM = True
try:
    os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')
    client = OpenAI(
        api_key=os.getenv("GROQ_API_KEY"),
        base_url="https://api.groq.com/openai/v1"
    )
    LLM_MODEL = "openai/gpt-oss-120b"  # pengganti resmi llama-3.3-70b-versatile
except Exception as e:
    USE_LLM = False
    print("LLM nonaktif (API key belum diset). Jawaban akan berupa konteks mentah.\n", e)

PROMPT_RAG = """Anda adalah asisten hukum untuk regulasi Kementerian Komunikasi dan Digital (Komdigi).
Jawab pertanyaan HANYA berdasarkan konteks regulasi berikut. Selalu sebutkan dasar hukumnya
(nama peraturan, pasal, ayat). Jika konteks tidak memadai, katakan tidak ditemukan.
KONTEKS:
{konteks}
PERTANYAAN: {pertanyaan}
JAWABAN (bahasa Indonesia formal, dengan sitasi pasal):"""

def rakit_konteks(df_hasil: pd.DataFrame) -> str:
    baris = []
    for _, r in df_hasil.iterrows():
        lok = f"Pasal {r['pasal']}" + (f" ayat ({r['ayat']})" if pd.notna(r.get('ayat')) else "")
        baris.append(f"[{r.get('nama_dokumen', r['chunk_id'])} — {lok}] {r['isi']}")
    return "\n\n".join(baris)

def jawab(pertanyaan: str, retriever, k: int = 8):
    t0 = time.time()
    hasil = retriever(pertanyaan, k)  # hasil harusnya DataFrame
    t_ret = time.time() - t0
    konteks = rakit_konteks(hasil)
    if USE_LLM:
        t0 = time.time()
        response = client.chat.completions.create(
            model=LLM_MODEL,
            messages=[
                {"role": "system", "content": "Anda adalah asisten hukum yang akurat dan tidak mengarang."},
                {
                    "role": "user",
                    "content": PROMPT_RAG.format(konteks=konteks, pertanyaan=pertanyaan)
                }
            ],
            temperature=0.1,
        )
        out = response.choices[0].message.content
        t_llm = time.time() - t0
    else:
        out, t_llm = "(LLM nonaktif — konteks teratas:)\n" + konteks[:1500], 0.0
    return {'jawaban': out, 'konteks': hasil, 'waktu_retrieval': t_ret, 'waktu_llm': t_llm}

res = jawab("Apa kewajiban penyelenggara penyiaran?", retrieve_dense)
print(res['jawaban'])
print(f"\n[retrieval {res['waktu_retrieval']:.2f}s | LLM {res['waktu_llm']:.2f}s]")

**Kewajiban penyelenggara penyiaran** (berdasarkan Undang‑Undang Nomor 32 Tahun 2002) antara lain:

1. **Mendapatkan izin penyelenggaraan penyiaran**  
   - *Pasal 33 ayat (1)*: “Sebelum menyelenggarakan kegiatannya lembaga penyiaran wajib memperoleh izin penyelenggaraan penyiaran.”

2. **Memenuhi ketentuan teknis penyiaran**  
   - *Pasal 32 ayat (1)*: “Setiap pendirian dan penyelenggaraan penyiaran wajib memenuhi ketentuan rencana dasar teknik penyiaran dan persyaratan teknis perangkat penyiaran.”

3. **Menunjuk penanggung jawab program**  
   - *Pasal 54*: “Pimpinan badan hukum lembaga penyiaran bertanggung jawab secara umum atas penyelenggaraan penyiaran dan wajib menunjuk penanggung jawab atas tiap‑tiap program yang dilaksanakan.”

4. **Memiliki kepedulian, pengetahuan, dan/atau pengalaman dalam bidang penyiaran**  
   - *Pasal 10 ayat (1)*: “memiliki kepedulian, pengetahuan dan/atau pengalaman dalam bidang penyiaran.”

5. **Bagi Lembaga Penyiaran Berlangganan, mematuhi ketentuan 

## §3 — Tahap 3: Hybrid Retrieval
Dense (FAISS) + sparse (BM25) digabung dengan **Reciprocal Rank Fusion (RRF)**,
ditambah filter metadata: jenis regulasi, tahun, bab, pasal, tipe norma, entitas, kata kunci.

In [ ]:
from rank_bm25 import BM25Okapi

STOPWORD = set("""yang dan di ke dari untuk pada dengan dalam adalah atau sebagaimana dimaksud
ayat pasal huruf angka bab ini itu oleh dapat tidak akan telah harus wajib sebagai atas terhadap
dilakukan berdasarkan sesuai antara serta bahwa maka setiap paling""".split())

def tokenisasi(t: str):
    return [w for w in re.findall(r'[a-z0-9]+', str(t).lower()) if w not in STOPWORD and len(w) > 2]

bm25 = BM25Okapi([tokenisasi(t) for t in korpus['teks_embed']])
print("BM25 siap:", len(korpus), "dokumen")

def retrieve_bm25(query: str, k: int = 10) -> pd.DataFrame:
    skor = bm25.get_scores(tokenisasi(query))
    top = np.argsort(skor)[::-1][:k]
    hasil = korpus.iloc[top].copy(); hasil['skor'] = skor[top]
    return hasil[['chunk_id','nama_dokumen','pasal','ayat','tipe','isi','skor']]

BM25 siap: 24294 dokumen


In [ ]:
def saring_metadata(df: pd.DataFrame, filter_meta: dict | None) -> pd.DataFrame:
    """filter_meta contoh: {'jenis':'UU', 'tahun':(2020,2026), 'tipe':['Sanksi','Larangan'],
                            'entitas':'Pemerintah', 'kata_kunci':'frekuensi', 'pasal':'34', 'bab':'BAB IV'}"""
    if not filter_meta: return df
    m = pd.Series(True, index=df.index)
    for kol, nilai in filter_meta.items():
        if kol == 'tahun' and isinstance(nilai, tuple):
            m &= df['tahun'].between(*nilai)
        elif kol in ('entitas', 'kata_kunci'):
            m &= df[kol].astype(str).str.contains(str(nilai), case=False, na=False)
        elif isinstance(nilai, (list, set, tuple)):
            m &= df[kol].isin(list(nilai))
        else:
            m &= df[kol].astype(str).str.lower() == str(nilai).lower()
    return df[m]

def retrieve_hybrid(query: str, k: int = 10, filter_meta: dict | None = None,
                    k_kandidat: int = 50, rrf_k: int = 60) -> pd.DataFrame:
    sub = saring_metadata(korpus, filter_meta)
    if len(sub) == 0: return sub
    # Dense — cari lebih banyak kandidat, lalu saring ke subset metadata
    qv = embedder.encode([query], normalize_embeddings=True).astype('float32')
    _, idx = faiss_index.search(qv, min(k_kandidat * 4, len(korpus)))
    rank_dense = [i for i in idx[0] if i in set(sub.index)][:k_kandidat]
    # Sparse
    skor_bm = bm25.get_scores(tokenisasi(query))
    rank_bm = [i for i in np.argsort(skor_bm)[::-1] if i in set(sub.index)][:k_kandidat]
    # RRF
    rrf = {}
    for peringkat, i in enumerate(rank_dense): rrf[i] = rrf.get(i, 0) + 1/(rrf_k + peringkat + 1)
    for peringkat, i in enumerate(rank_bm):    rrf[i] = rrf.get(i, 0) + 1/(rrf_k + peringkat + 1)
    top = sorted(rrf, key=rrf.get, reverse=True)[:k]
    hasil = korpus.loc[top].copy(); hasil['skor'] = [rrf[i] for i in top]
    return hasil[['chunk_id','nama_dokumen','pasal','ayat','tipe','isi','skor']]

# Contoh: hanya cari norma Sanksi pada regulasi tahun ≥ 2020
retrieve_hybrid("denda administratif penggunaan spektrum frekuensi radio",
                k=5, filter_meta={'tipe': ['Sanksi'], 'tahun': (2020, 2026)})

,chunk_id,nama_dokumen,pasal,ayat,tipe,isi,skor
6485,PP_46_2021__P54__a1__hb,PP Nomor 46 Tahun 2021,54,1,Sanksi,denda administratif; dan/atau,0.032787
6490,PP_46_2021__P54__a6__1,PP Nomor 46 Tahun 2021,54,6,Sanksi,Denda administratif sebagaimana dimaksud pada ayat,0.032258
6489,PP_46_2021__P54__a6__0,PP Nomor 46 Tahun 2021,54,6,Sanksi,Denda,0.031498
5462,PP_28_2025__P493__a1,PP Nomor 28 Tahun 2025,493,1,Sanksi,Setiap pemegang lzin pita frekuensi radio yang berdasarkan hasil Pengawasan ditemukan melakukan pengalihan hak pengg...,0.029857
5458,PP_28_2025__P492__a1,PP Nomor 28 Tahun 2025,492,1,Sanksi,Setiap pemegang lzin pita frekuensi radio yang berdasarkan hasil Pengawasan ditemukan melakukan kerja sama penggunaa...,0.028446


## §4 — Tahap 4: Knowledge Graph
Graf berarah dengan simpul: **dokumen → bab → pasal → satuan (chunk)**, plus simpul **entitas**.
Sisi rujukan silang diambil dari hasil ekstraksi Tahap 1 (`ref_internal`, `ref_eksternal`).

In [ ]:
KG = nx.MultiDiGraph()

def id_bab(doc, bab):     return f"{doc}::{bab}"
def id_pasal(doc, pasal): return f"{doc}::P{pasal}"

# 1) Hirarki struktural
for _, r in korpus.iterrows():
    d, b, p = r['doc_id'], id_bab(r['doc_id'], r['bab']), id_pasal(r['doc_id'], r['pasal'])

    # Memastikan 'tahun' adalah nilai numerik sebelum konversi ke int
    tahun_value = int(r['tahun']) if pd.notna(r['tahun']) else None

    KG.add_node(d, tipe='dokumen', label=r['nama_dokumen'], jenis=r['jenis'], tahun=tahun_value)
    KG.add_node(b, tipe='bab', label=str(r['nama_bab']))
    KG.add_node(p, tipe='pasal', label=f"Pasal {r['pasal']}")
    KG.add_node(r['chunk_id'], tipe='satuan', norma=r['tipe'], pasal=str(r['pasal']))
    KG.add_edge(d, b, rel='memuat'); KG.add_edge(b, p, rel='memuat')
    KG.add_edge(p, r['chunk_id'], rel='memuat')

# 2) Rujukan internal antar-pasal (hanya yang valid)
for _, r in ref_internal[ref_internal.valid].iterrows():
    KG.add_edge(id_pasal(r['doc_id'], r['pasal_asal']),
                id_pasal(r['doc_id'], r['pasal_tuju']), rel='merujuk')

# 3) Rujukan antar-dokumen (yang tersedia dalam korpus)
for _, r in ref_eksternal[ref_eksternal.dalam_korpus].drop_duplicates(['doc_id','doc_tuju']).iterrows():
    if r['doc_id'] != r['doc_tuju']:
        KG.add_edge(r['doc_id'], r['doc_tuju'], rel='merujuk_dokumen')

# 4) Entitas
for _, r in korpus[korpus['entitas'].notna()].iterrows():
    for ent in str(r['entitas']).split(','):
        ent = ent.strip()
        if ent:
            eid = f"ENT::{ent}"
            KG.add_node(eid, tipe='entitas', label=ent)
            KG.add_edge(r['chunk_id'], eid, rel='menyebut')

print(f"Simpul: {KG.number_of_nodes():,} | Sisi: {KG.number_of_edges():,}")
print(pd.Series([d['tipe'] for _, d in KG.nodes(data=True)]).value_counts().to_string())
print()
print(pd.Series([d['rel'] for _, _, d in KG.edges(data=True)]).value_counts().to_string())

with open(os.path.join(IFACT_DIR, 'knowledge_graph.gpickle'), 'wb') as f:
    pickle.dump(KG, f)

Simpul: 26,992 | Sisi: 89,733
satuan     24294
pasal       2443
bab          195
entitas       36
dokumen       24

memuat             72882
menyebut           10410
merujuk             6426
merujuk_dokumen       15


In [ ]:
# Statistik graf untuk laporan tesis + visualisasi sampel
deg = pd.Series({n: KG.degree(n) for n, d in KG.nodes(data=True) if d['tipe'] == 'pasal'})
print("Pasal paling terhubung (hub regulasi):")
for n, v in deg.sort_values(ascending=False).head(8).items():
    print(f"  {n}  (derajat {v})")

# Visualisasi interaktif subgraf sekitar satu pasal (buka file HTML hasilnya)
from pyvis.network import Network
def gambar_subgraf(node_pusat: str, radius: int = 1, out='subgraf.html'):
    simpul = {node_pusat}
    for _ in range(radius):
        tetangga = set()
        for s in simpul:
            tetangga |= set(KG.successors(s)) | set(KG.predecessors(s))
        simpul |= tetangga
    sub = KG.subgraph(simpul)
    warna = {'dokumen':'#d62728','bab':'#ff7f0e','pasal':'#1f77b4','satuan':'#2ca02c','entitas':'#9467bd'}
    net = Network(height='600px', directed=True, notebook=False, cdn_resources='remote')
    for n, d in sub.nodes(data=True):
        net.add_node(n, label=d.get('label', n)[:40], color=warna[d['tipe']], title=d['tipe'])
    for u, v, d in sub.edges(data=True):
        net.add_edge(u, v, label=d['rel'])
    net.save_graph(out); print("Tersimpan:", out)

gambar_subgraf(deg.idxmax(), radius=1)

Pasal paling terhubung (hub regulasi):
  UU_6_2023::P1  (derajat 1150)
  Permen_8_2026::P9  (derajat 669)
  UU_6_2023::P15  (derajat 380)
  UU_6_2023::P26  (derajat 372)
  UU_6_2023::P5  (derajat 351)
  UU_6_2023::P7  (derajat 338)
  Permen_2_2025::P1  (derajat 326)
  UU_6_2023::P40  (derajat 295)
Tersimpan: subgraf.html


## §5 — Tahap 5: Graph RAG
Alur: **hybrid retrieval → seed chunks → traversal graf** untuk memperluas konteks dengan:
1. saudara satu pasal (koherensi struktural),
2. pasal yang dirujuk / merujuk (penalaran silang),
3. satuan lain yang berbagi entitas.

Skor ekspansi diberi peluruhan (*decay*) agar konteks hasil traversal tidak menenggelamkan hasil semantik.

In [ ]:
PETA_CHUNK = korpus.set_index('chunk_id')

def perluas_graf(seed_ids: list[str], bobot_seed: dict,
                 maks_per_seed: int = 4, decay: float = 0.5) -> dict:
    skor = dict(bobot_seed)
    for cid in seed_ids:
        if cid not in KG: continue
        kandidat = []
        # (a) saudara satu pasal
        for p in KG.predecessors(cid):
            if KG.nodes[p]['tipe'] == 'pasal':
                kandidat += [(c, 1.0) for c in KG.successors(p)
                             if KG.nodes[c]['tipe'] == 'satuan' and c != cid]
                # (b) pasal dirujuk / merujuk
                pasal_terkait = ([v for _, v, d in KG.out_edges(p, data=True) if d['rel']=='merujuk'] +
                                 [u for u, _, d in KG.in_edges(p, data=True)  if d['rel']=='merujuk'])
                for pt in pasal_terkait:
                    kandidat += [(c, 0.8) for c in KG.successors(pt)
                                 if KG.nodes[c]['tipe'] == 'satuan']
        # (c) berbagi entitas
        for e in KG.successors(cid):
            if KG.nodes[e]['tipe'] == 'entitas':
                kandidat += [(c, 0.5) for c in KG.predecessors(e)
                             if KG.nodes[c]['tipe'] == 'satuan' and c != cid]
        # ambil terbaik per seed
        terbaik = {}
        for c, w in kandidat: terbaik[c] = max(terbaik.get(c, 0), w)
        for c, w in sorted(terbaik.items(), key=lambda x: -x[1])[:maks_per_seed]:
            tambahan = bobot_seed[cid] * decay * w
            skor[c] = max(skor.get(c, 0), tambahan)
    return skor

def retrieve_graphrag(query: str, k: int = 10, k_seed: int = 6,
                      filter_meta: dict | None = None) -> pd.DataFrame:
    seed = retrieve_hybrid(query, k=k_seed, filter_meta=filter_meta)
    if len(seed) == 0: return seed
    bobot = dict(zip(seed['chunk_id'], seed['skor'] / seed['skor'].max()))
    skor = perluas_graf(seed['chunk_id'].tolist(), bobot)
    urut = sorted(skor, key=skor.get, reverse=True)[:k]
    hasil = PETA_CHUNK.loc[urut].reset_index()
    hasil['skor'] = [skor[c] for c in urut]
    hasil['asal'] = ['seed' if c in set(seed['chunk_id']) else 'graf' for c in urut]
    return hasil[['chunk_id','nama_dokumen','pasal','ayat','tipe','isi','skor','asal']]

retrieve_graphrag("sanksi administratif bagi penyelenggara yang tidak memenuhi kewajiban perizinan", k=8)

,chunk_id,nama_dokumen,pasal,ayat,tipe,isi,skor,asal
0,Permen_5_2021__P237__a1,Permen Nomor 5 Tahun 2021,237,1,Sanksi,"Direksi, pengurus, perorangan, dan/atau badan hukum Pelaku Usaha dapat ditetapkan dalam Daftar Hitam Penyelenggara d...",1.000000,seed
1,Permen_5_2021__P225__a8,Permen Nomor 5 Tahun 2021,225,8,Sanksi,"Pengenaan sanksi administratif, selain pencabutan layanan dan/atau Perizinan Berusaha, tidak menghilangkan kewajiban...",0.995360,seed
2,PP_28_2025__P531__a8,PP Nomor 28 Tahun 2025,531,8,Sanksi,Penyelenggara sertifikasi elektronik dikenai sanksi administratif berupa penghentian sementara kegiatan penyelenggar...,0.903971,seed
3,Permen_5_2021__P227,Permen Nomor 5 Tahun 2021,227,<NA>,Sanksi,Pengenaan sanksi administratif tehadap pelanggaran kewajiban Pelaku Usaha untuk:,0.891784,seed
4,UU_38_2009__P40__0,UU Nomor 38 Tahun 2009,40,<NA>,Sanksi,Penyelenggara Pos yang dengan sengaja dan tanpa hak tidak menjaga keamanan dan keselamatan kiriman sebagaimana dimak...,0.884891,seed
5,Permen_5_2021__P229__a1,Permen Nomor 5 Tahun 2021,229,1,Sanksi,Direktur menerbitkan teguran tertulis sebagaimana dimaksud dalam Pasal 225 ayat (3) huruf a bagi Pelaku Usaha yang m...,0.816950,seed
6,Permen_5_2021__P237__a2,Permen Nomor 5 Tahun 2021,237,2,Larangan,"Direksi, pengurus, perorangan, dan/atau badan hukum Pelaku Usaha yang ditetapkan dalam Daftar Hitam Penyelenggara, d...",0.500000,graf
7,Permen_5_2021__P237__a3,Permen Nomor 5 Tahun 2021,237,3,Ketentuan,"Direksi, pengurus, perorangan, dan/atau badan hukum Pelaku Usaha dapat dikeluarkan dari Daftar Hitam Penyelenggara s...",0.500000,graf


In [ ]:
# Pipeline lengkap Graph-RAG dengan LLM
res = jawab("Bagaimana ketentuan penggunaan spektrum frekuensi radio tanpa izin dan sanksinya?",
            retrieve_graphrag, k=8)
print(res['jawaban'])
print(f"\n[retrieval {res['waktu_retrieval']:.2f}s | LLM {res['waktu_llm']:.2f}s]")
res['konteks']

Berdasarkan konteks regulasi yang diberikan, informasi mengenai ketentuan penggunaan spektrum frekuensi radio tanpa izin beserta sanksinya **tidak ditemukan**.

[retrieval 45.74s | LLM 5.42s]


,chunk_id,nama_dokumen,pasal,ayat,tipe,isi,skor,asal
0,PP_46_2021__P64__a1__he,PP Nomor 46 Tahun 2021,64,1,Ketentuan,mengalihkan izin penggunaan Spektrum Frekuensi Radio tanpa persetujuan Menteri;,1.000000,seed
1,Permen_2_2023__P15__a1__ha,Permen Nomor 2 Tahun 2023,15,1,Ketentuan,tidak mendapatkan proteksi dari penggunaan Spektrum Frekuensi Radio oleh pemegang izin penggunaan Spektrum Frekuensi...,0.901677,seed
2,Permen_2_2023__P12__a1__hc,Permen Nomor 2 Tahun 2023,12,1,Ketentuan,tidak mendapatkan proteksi dari penggunaan Spektrum Frekuensi Radio oleh pemegang izin penggunaan Spektrum Frekuensi...,0.898547,seed
3,PP_46_2021__P63__a1__hb,PP Nomor 46 Tahun 2021,63,1,Ketentuan,permohonan penghentian izin penggunaan -Spektrum Frekuensi Radio oleh pemegang izin penggunaan Spektrum Frekuensi Ra...,0.871176,seed
4,Permen_9_2023__P3__a2,Permen Nomor 9 Tahun 2023,3,2,Kewajiban,Kewajiban BHP Spektrum Frekuensi Radio sebagaimana dimaksud pada ayat (1) mulai dikenakan pada saat izin penggunaan ...,0.826046,seed
5,Permen_9_2023__P3__a1,Permen Nomor 9 Tahun 2023,3,1,Kewajiban,Pemegang izin penggunaan Spektrum Frekuensi Radio wajib membayar BHP Spektrum Frekuensi Radio.,0.809783,seed
6,PP_46_2021__P64__a1,PP Nomor 46 Tahun 2021,64,1,Ketentuan,Pengakhiran masa laku izin penggunaan Spektrum Frekuensi Radio atas dasar pencabutan sebagaimana dimaksud dalam Pasa...,0.500000,graf
7,PP_46_2021__P64__a1__ha,PP Nomor 46 Tahun 2021,64,1,Ketentuan,ain Penyelenggaraan Telekomunikasi atau IPP telah berakhir atau dicabuU,0.500000,graf


## §6 — Tahap 6: Evaluasi
Tiga sistem dibandingkan: **Dense (baseline)**, **Hybrid**, **Graph-RAG**.

**Gold standard:** isi file `gold_standard.csv` (kolom: `pertanyaan`, `relevan` berisi
daftar `chunk_id` dipisah `;`). Templat dibuat otomatis di bawah — lengkapi manual
berdasarkan penelusuran Anda terhadap regulasi (disarankan ≥ 30–50 pertanyaan untuk tesis).

In [ ]:
GOLD_PATH = os.path.join(BASE_DIR, 'gold_standard.csv')

if not os.path.exists(GOLD_PATH):
    contoh = pd.DataFrame({
        'pertanyaan': [
            'Apa sanksi bagi penyelenggara telekomunikasi tanpa izin?',
            'Apa kewajiban penyelenggara pos terhadap kerahasiaan kiriman?',
            'Bagaimana pengaturan penggunaan spektrum frekuensi radio?',
        ],
        'relevan': ['ISI_CHUNK_ID_1;ISI_CHUNK_ID_2', 'ISI_CHUNK_ID_3', 'ISI_CHUNK_ID_4;ISI_CHUNK_ID_5'],
    })
    contoh.to_csv(GOLD_PATH, index=False)
    print("Templat gold standard dibuat →", GOLD_PATH)
    print("Gunakan retrieve_hybrid()/pencarian manual untuk menemukan chunk_id relevan, lalu isi file tsb.")
else:
    gold = pd.read_csv(GOLD_PATH)
    gold = gold[~gold['relevan'].str.contains('ISI_CHUNK_ID', na=True)]
    print(f"Gold standard dimuat: {len(gold)} pertanyaan")

Gold standard dimuat: 0 pertanyaan


In [ ]:
def evaluasi_retrieval(gold: pd.DataFrame, sistem: dict, k_list=(3, 5, 10)) -> pd.DataFrame:
    baris = []
    for nama, retriever in sistem.items():
        for _, g in gold.iterrows():
            relevan = set(str(g['relevan']).split(';'))
            t0 = time.time()
            hasil = retriever(g['pertanyaan'], max(k_list))['chunk_id'].tolist()
            waktu = time.time() - t0
            # MRR
            rr = 0.0
            for pos, cid in enumerate(hasil, 1):
                if cid in relevan: rr = 1/pos; break
            rec = {'sistem': nama, 'pertanyaan': g['pertanyaan'], 'MRR': rr, 'waktu_s': waktu}
            for k in k_list:
                topk = set(hasil[:k])
                rec[f'Recall@{k}']    = len(topk & relevan) / len(relevan)
                rec[f'Precision@{k}'] = len(topk & relevan) / k
                rec[f'Hit@{k}']       = float(len(topk & relevan) > 0)
            baris.append(rec)
    df = pd.DataFrame(baris)
    return df.groupby('sistem').mean(numeric_only=True).round(4)

SISTEM = {
    'Dense (baseline)': retrieve_dense,
    'Hybrid (RRF)'    : retrieve_hybrid,
    'Graph-RAG'       : retrieve_graphrag,
}

if 'gold' in dir() and len(gold):
    tabel = evaluasi_retrieval(gold, SISTEM)
    display(tabel)
    tabel.to_csv(os.path.join(ARTIFACT_DIR, 'hasil_evaluasi_retrieval.csv'))
else:
    print("Lengkapi gold_standard.csv terlebih dahulu, lalu jalankan ulang sel ini.")

Lengkapi gold_standard.csv terlebih dahulu, lalu jalankan ulang sel ini.


RAGAS Error

In [ ]:
# (Opsional) Evaluasi kualitas jawaban dengan RAGAS
# Metrik: Faithfulness, Context Precision, Context Recall, Answer Relevancy.
# RAGAS memerlukan LLM sebagai juri (OpenAI/Gemini via LangChain). Kerangka minimal:
#
# NOTE: Evaluasi RAGAS saat ini tidak dapat dilakukan karena konflik dependensi yang tidak terselesaikan.
# Library 'ragas' secara internal mencoba mengimpor 'langchain_community.chat_models.vertexai',
# yang menyebabkan ModuleNotFoundError karena konflik protobuf dengan paket-paket Google Cloud lainnya.
# Sel ini dinonaktifkan sementara. Silakan pertimbangkan alternatif evaluasi.
#

!pip install -q ragas datasets google-generativeai langchain-google-genai langchain_community langchain-openai

from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
from datasets import Dataset
from langchain_openai import ChatOpenAI
import os
from google.colab import userdata

# # Periksa apakah 'gold' DataFrame ada dan tidak kosong
if 'gold' in locals() and len(gold) > 0:
#     # Konfigurasi OpenAI (Groq) untuk RAGAS
#     # Kunci API dan base URL untuk Groq sudah diatur di sel sebelumnya
#     # (AQk9N4tnYNT1), langchain-openai akan otomatis mengambil dari env vars.
#     # Namun, untuk eksplisit dan kejelasan, kita bisa mengulang set environment vars
#     # atau memastikan mereka ada.
    os.environ["OPENAI_API_KEY"] = userdata.get('GROQ_API_KEY')
    os.environ["OPENAI_API_BASE"] = "https://api.groq.com/openai/v1"

    ragas_llm = ChatOpenAI(model_name="openai/gpt-oss-120b", temperature=0.1)

    rekaman = []
    for _, g in gold.iterrows():
        r = jawab(g['pertanyaan'], retrieve_graphrag, k=8)
        rekaman.append({'question': g['pertanyaan'],
                         'answer': r['jawaban'],
                         'contexts': r['konteks']['isi'].tolist(),
                         'ground_truth': ''})   # isi jawaban referensi bila ada

    hasil_ragas = evaluate(Dataset.from_list(rekaman),
                           metrics=[faithfulness, answer_relevancy, context_precision, context_recall],
                           llm=ragas_llm)
    print(hasil_ragas)
else:
    print("Gold standard kosong. Harap lengkapi `gold_standard.csv` terlebih dahulu sebelum menjalankan evaluasi RAGAS.")

# Menampilkan pesan bahwa RAGAS dinonaktifkan
print("Evaluasi RAGAS dinonaktifkan karena masalah dependensi yang tidak terselesaikan. Silakan lengkapi `gold_standard.csv` dan jalankan bagian lain dari notebook.")

ModuleNotFoundError: No module named 'langchain_community.chat_models.vertexai'

## Alur kerja saat korpus bertambah
1. Tambahkan `korpusXXX.csv` (skema sama) ke folder `korpus` di Drive.
2. Jalankan ulang **§1** — validasi otomatis melaporkan rujukan luar korpus yang paling sering muncul
   (prioritas regulasi berikutnya untuk ditambahkan).
3. Jalankan **§2** (indeks otomatis dibangun ulang karena ukuran korpus berubah), **§3** (BM25), **§4** (graf).
4. Jalankan ulang **§6** untuk melihat dampak perluasan korpus pada metrik.

**Ide analisis untuk bab pembahasan tesis:** ablasi (dense vs hybrid vs graph),
kontribusi tiap jenis sisi graf (matikan `merujuk` / `menyebut` bergantian),
sensitivitas parameter `decay` dan `maks_per_seed`, serta perbandingan model embedding
(MiniLM vs multilingual-e5).